P36 – Loan Approved or Not?

Decision Tree Classification Project
Student details:
Name      :Samar rathod
Roll No.  :B-42
Div       :D
Enrollment: 24004501210359

Problem statement
Predict whether a loan application is approved (`Loan_Status`) using applicant information such as gender, marital status, dependents,education, 
employment, income, loan amount, loan term, credit history, and property area

#IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay,roc_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Required libraries imported successfully!")

print("All libraries imported successfully!")

#LOAD THE DATASET

In [ ]:
df = pd.read_csv("train.csv")

print("Dataset shape:", df.shape)
display(df.head())

Understand the Data and Create a Data Dictionary

In [ ]:
print("Dataset information:")
df.info()

print("\nColumn names:")
print(df.columns.tolist())

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nData types:")
print(df.dtypes)

data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [str(df[col].dtype) for col in df.columns],
    "Missing Values": [df[col].isnull().sum() for col in df.columns],
    "Unique Values": [df[col].nunique() for col in df.columns]
})

display(data_dictionary)

Prepare the target column (Y)

In [ ]:
print("Loan Status counts:")
print(df["Loan_Status"].value_counts())

print("\nLoan Status percentages:")
print((df["Loan_Status"].value_counts(normalize=True) * 100).round(2))

sns.countplot(data=df, x="Loan_Status")
plt.title("Loan Approval Status")
plt.xlabel("Loan Status (Y = Approved, N = Not Approved)")
plt.ylabel("Number of Applicants")
plt.show()

Clean Missing Values and Create Total Income

In [ ]:
df = df.drop_duplicates().copy()

# Remove the ID column because it does not describe applicant eligibility
if "Loan_ID" in df.columns:
    df.drop(columns=["Loan_ID"], inplace=True)

# Combine applicant income and co-applicant income
df["TotalIncome"] = (
    df["ApplicantIncome"] + df["CoapplicantIncome"]
)

# Standardize the target labels
df["Loan_Status"] = df["Loan_Status"].astype(str).str.strip().str.upper()

# Remove rows without a valid target
df = df[df["Loan_Status"].isin(["Y", "N"])].copy()

# Convert categorical yes/no values into consistent labels
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Missing values before feature imputation:")
print(df.isnull().sum())

print("\nCleaned dataset shape:", df.shape)
display(df.head())

Exploratory Data Analysis (EDA) and Five Insights

In [ ]:
# 1. Loan status distribution
sns.countplot(data=df, x="Loan_Status")
plt.title("Loan Approval Distribution")
plt.show()

# 2. Applicant income distribution
sns.histplot(data=df, x="ApplicantIncome", kde=True)
plt.title("Applicant Income Distribution")
plt.show()

# 3. Total income by loan status
sns.boxplot(data=df, x="Loan_Status", y="TotalIncome")
plt.title("Total Income vs Loan Status")
plt.show()

# 4. Credit history vs loan status
sns.countplot(data=df, x="Credit_History", hue="Loan_Status")
plt.title("Credit History vs Loan Status")
plt.show()

# 5. Education vs loan status
sns.countplot(data=df, x="Education", hue="Loan_Status")
plt.title("Education vs Loan Status")
plt.show()

# Numeric correlation
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

Encode Categorical Data and Select Features

In [ ]:
# Separate input features and target
X = df.drop(columns=["Loan_Status"])
y = df["Loan_Status"].map({"N": 0, "Y": 1})

# Convert categorical variables into numeric dummy columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Ensure all feature values are numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Replace infinite values with missing values
X = X.replace([np.inf, -np.inf], np.nan)

print("Encoded feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())

Split the Dataset into Training and Testing Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Train the Baseline Decision Tree Model

In [ ]:
# Impute missing values using training data only
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Select the best features using training data only
k = min(10, X_train_imputed.shape[1])

selector = SelectKBest(score_func=f_classif, k=k)

X_train_selected = selector.fit_transform(
    X_train_imputed, y_train
)

X_test_selected = selector.transform(X_test_imputed)

selected_features = X_train_imputed.columns[
    selector.get_support()
].tolist()

print("Selected features:")
print(selected_features)

# Train the baseline Decision Tree
baseline_model = DecisionTreeClassifier(
    random_state=42
)

baseline_model.fit(X_train_selected, y_train)

y_pred = baseline_model.predict(X_test_selected)

print("\nBaseline Model Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Not Approved", "Approved"],
    zero_division=0
))

Compare Gini, Entropy, and Different Tree Depths

In [ ]:
results = []

for criterion in ["gini", "entropy"]:
    for depth in [2, 3, 5, 7, 9, None]:

        model = DecisionTreeClassifier(
            criterion=criterion,
            max_depth=depth,
            random_state=42
        )

        model.fit(X_train_selected, y_train)
        predictions = model.predict(X_test_selected)

        results.append({
            "Criterion": criterion,
            "Max Depth": str(depth),
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1 Score": f1_score(
                y_test, predictions, zero_division=0
            )
        })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by="F1 Score", ascending=False
    ).round(3)
)

Compare Results and Select the Final Model

In [ ]:
# Select the model with the highest F1 score
best_row = results_df.sort_values(
    by="F1 Score", ascending=False
).iloc[0]

best_criterion = best_row["Criterion"]
best_depth = best_row["Max Depth"]

if best_depth == "None":
    best_depth = None
else:
    best_depth = int(best_depth)

final_model = DecisionTreeClassifier(
    criterion=best_criterion,
    max_depth=best_depth,
    random_state=42
)

final_model.fit(X_train_selected, y_train)

final_pred = final_model.predict(X_test_selected)

print("Selected criterion:", best_criterion)
print("Selected max depth:", best_depth)

print("\nFinal Model Evaluation")
print("Accuracy:", round(accuracy_score(y_test, final_pred), 3))
print("Precision:", round(precision_score(y_test, final_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, final_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, final_pred, zero_division=0), 3))

# Confusion matrix
cm = confusion_matrix(y_test, final_pred, labels=[0, 1])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Approved", "Approved"],
    yticklabels=["Not Approved", "Approved"]
)

plt.title("Confusion Matrix – Final Decision Tree")
plt.xlabel("Predicted Status")
plt.ylabel("Actual Status")
plt.show()

# ROC curve and AUC
if len(np.unique(y_test)) == 2:
    y_probability = final_model.predict_proba(X_test_selected)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_test, y_probability)
    auc_value = roc_auc_score(y_test, y_probability)

    plt.plot(fpr, tpr, label=f"AUC = {auc_value:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve – Final Decision Tree")
    plt.legend()
    plt.show()

# Visualize the decision tree
plt.figure(figsize=(20, 10))

plot_tree(
    final_model,
    feature_names=selected_features,
    class_names=["Not Approved", "Approved"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Final Loan Approval Decision Tree")
plt.show()

# Feature importance
importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df)

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")
plt.show()

Predict Loan Approval for Three Applications

In [ ]:
# Take three applications from the test dataset
three_applications = X_test_imputed.iloc[:3]

# Apply the same feature selector used during training
three_selected = selector.transform(three_applications)

# Predict loan status and probability
predictions = final_model.predict(three_selected)
probabilities = final_model.predict_proba(three_selected)[:, 1]

prediction_results = pd.DataFrame({
    "Application": ["Application 1", "Application 2", "Application 3"],
    "Predicted Loan Status": [
        "Approved" if p == 1 else "Not Approved"
        for p in predictions
    ],
    "Approval Probability (%)": (probabilities * 100).round(2)
})

display(prediction_results)

Conclusion and Prepare the PDF Report

In [ ]:
print("P36 – Loan Approved or Not?")

print("\nConclusion:")
print(
    "A Decision Tree classification model was developed to predict "
    "loan approval using applicant information."
)

print("\nSelected criterion:", best_criterion)
print("Selected maximum depth:", best_depth)
print("Accuracy:", round(accuracy_score(y_test, final_pred), 3))
print(
    "Precision:",
    round(precision_score(y_test, final_pred, zero_division=0), 3)
)
print(
    "Recall:",
    round(recall_score(y_test, final_pred, zero_division=0), 3)
)
print(
    "F1 Score:",
    round(f1_score(y_test, final_pred, zero_division=0), 3)
)

print("\nSelected features:", selected_features)

print(
    "\nThe model's predictions are based on historical data and should "
    "not be used as the sole basis for real loan decisions."
)

**Conclusion:**
The Decision Tree model predicts whether a loan may be approved or rejected. It achieved 85.4% accuracy in the saved results. I learned how to clean data, train a machine learning model, and evaluate its performance. The model can make mistakes, so it should not be the only basis for loan approval.
